# Modül 2 — Veri + ML Baseline (Pandas → EDA → sklearn)

**Durum:** 0/30 egzersiz, 6 oturum. Çıktı: **RND-002**, yani kendi görsel veri setinin denetimi (audit) ve sızıntısız bir sklearn baseline'ı.

**Bu modülün eski Month 2'den farkı:**

- Titanic yerine **kendi görsellerin** kullanılıyor. Pandas bir görsel *manifest*'i üzerinde öğreniliyor (dosya yolu, etiket, bina, kalite ölçüleri). Bu, ileride PyTorch `Dataset`'inin okuyacağı tablonun aynısı.
- Split, temizlik ve modellemeden **önce** yapılıyor. İmputer ve scaler gibi her şey `Pipeline` içinde ve sadece train'e fit ediliyor.
- Aynı binanın görselleri hem train'de hem test'te olamaz. Bunun için `GroupShuffleSplit` / `GroupKFold` kullanılıyor.
- Model seçimi CV ile yapılıyor; test seti en sonda, **bir kez** kullanılıyor.
- Eklenenler: PR eğrisi (Modül 4'teki mAP'in temeli), `DummyClassifier` tabanı, `class_weight`.

**Veri hazırlığı (Oturum 1'den önce, kodsuz):**

```text
learning/data/m2_images/
├── <sinif_a>/   örn. facade
├── <sinif_b>/   örn. interior
└── <sinif_c>/   örn. detail
```

- Sınıf başına en az 30–50 görsel olsun, toplam en az 100.
- Dosya adı binayı söylesin: `<bina-id>_<no>.jpg` (örn. `kadikoy-ev_03.jpg`). Bina id'si group split için gerekiyor.
- Elinde yeterli görsel yoksa alternatif: ImageToRender'ın tarihi yapı elemanı sınıfları. Ya da Kaggle'da açık bir mimari stil veri seti; ama onlarda bina id'si olmayabilir, o zaman group split'i dosya adından üretemezsin.
- Bilerek 1–2 bozuk dosya ekle (boş bir `.jpg`) ve 2–3 kopya görsel koy. Denetim egzersizleri bunları yakalamalı.

**Isınma (isteğe bağlı, 1–2 oturum):** Kaggle Learn *Pandas* kursu.

**Nasıl kullanılır:** 1 oturum = 5 egzersiz. Önce kendin yaz, sonucu `print` et veya göster, altına bir cümle gözlem ekle. AI sadece ipucu ve inceleme için.

In [ ]:
%matplotlib inline
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

DATA_DIR = Path("data/m2_images")
OUTPUT_DIR = Path("outputs/m2")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42
IMG_EXTS = {".jpg", ".jpeg", ".png"}

print(pd.__version__, cv2.__version__)

---
## Oturum 1 — Manifest + Pandas temel

Manifest, her satırı bir görsel olan bir tablodur. Bu modüldeki her şey bu tablo üzerinden yapılır.

### Exercise 1 — Build the manifest

Walk `DATA_DIR` with `pathlib` (`rglob`), keep only `IMG_EXTS`.

Build a DataFrame with columns: `path`, `filename`, `label` (parent folder name), `group` (building id parsed from the filename).

Print `.head()` and `len(df)`.

In [ ]:
rows = []
# TODO: fill rows with dicts, then
# df = pd.DataFrame(rows)


### Exercise 2 — Read image metadata safely

Write a function `read_meta(path) -> dict` that returns `height`, `width`, `channels`, `file_kb`, `ok`.

If `cv2.imread` returns `None`, return `ok=False` and `NaN` for the image fields — **do not crash**.

Apply it to every row and join the result to `df` (`pd.DataFrame(list_of_dicts)` + `pd.concat(axis=1)`).

In [ ]:
# TODO


### Exercise 3 — Inspect

Print `.info()`, `.describe()`, `.dtypes`, and `df['ok'].value_counts()`.

Write which columns are numeric, which categorical, and which look suspicious.

In [ ]:
# TODO


### Exercise 4 — loc / iloc / boolean filtering

- rows 2–4 with `.iloc`
- all broken files with `.loc` + a mask
- images narrower than 512 px **and** of one chosen label (use `&`)
- one cell: the `width` of the 5th row

Comment: when do you prefer `loc` over `iloc`?

In [ ]:
# TODO


### Exercise 5 — Sorting and counting

- sort by `file_kb` descending, then by `label` + `width`
- `nlargest(3, 'width')`
- `value_counts()` of `label` and of `group`

How many buildings does each class contain? (`groupby('label')['group'].nunique()`)

In [ ]:
# TODO


---
## Oturum 2 — Kalite özellikleri, kopyalar, merge, eksik değerler

Bu özellikler görsel başına ayrı ayrı hesaplanır (`fit` yok), bu yüzden split'ten önce hesaplamak **sızıntı yaratmaz**. Sızıntı, veri setinin tamamından istatistik öğrenen adımlarda olur (ortalama ile doldurma, scaler gibi). O adımlar Oturum 4'te.

### Exercise 6 — GroupBy aggregation

Group by `label`: compute `count`, `mean` and `std` of `width`, `height`, `file_kb` with `.agg([...])`. Sort by count.

One sentence: is any class systematically larger or smaller in resolution? Why could that become a shortcut for a model?

In [ ]:
# TODO


### Exercise 7 — Quality features with apply

Write `quality(path) -> dict` returning:

- `blur` = variance of `cv2.Laplacian(gray, cv2.CV_64F)` (low = blurry)
- `brightness` = mean gray value
- `edge_density` = share of non-zero pixels in `cv2.Canny(gray, 50, 150)`

Add them as columns (skip `ok == False` rows). This reuses Modül 1.

In [ ]:
# TODO


### Exercise 8 — Duplicate detection

Implement an **average hash**: resize gray image to 8×8, compare each pixel to the mean → 64 booleans → a hex string.

Add a `ahash` column; find rows where `df.duplicated('ahash', keep=False)`. Show the duplicate pairs side by side.

Why is a duplicate across train and test a form of leakage?

In [ ]:
# TODO


### Exercise 9 — Merge

Create a small second table by hand: `filename`, `note` (e.g. 'night', 'scaffolding', 'cropped') for ~10 images.

Merge it with `how='inner'` and `how='left'`. Compare row counts and explain what happens to images without a note.

In [ ]:
# TODO


### Exercise 10 — Missing values — decide, don't just fill

Print missing count and % per column. Broken files produce NaN in every image column.

Decide per column: drop rows / drop column / impute **later inside the Pipeline**. Write the decision as a small table in a markdown cell.

Then create `df_clean` (broken files and duplicates removed — keep one of each duplicate group).

In [ ]:
# TODO


---
## Oturum 3 — EDA ve veri seti kartı

### Exercise 11 — Class balance

`value_counts(normalize=True)` of `label` + a bar chart.

Balanced, mildly or severely imbalanced? Which metric would accuracy hide here?

In [ ]:
# TODO


### Exercise 12 — Distributions per class

For `blur` and `brightness`: histogram + boxplot **per label** (`df.boxplot(column=..., by='label')`).

Which feature differs most between classes?

In [ ]:
# TODO


### Exercise 13 — Outliers with IQR — then look at them

On `blur`: compute Q1, Q3, IQR, bounds, count outliers.

Show the outlier **images** in a grid (`plt.subplots`). Decide: drop, keep, or flag — and why. Looking at the images is the point.

In [ ]:
# TODO


### Exercise 14 — Correlation

`.corr(numeric_only=True)` → `plt.imshow` heatmap with labels.

Name the two most correlated features. Which one would you drop if you had to, and why?

In [ ]:
# TODO


### Exercise 15 — Dataset card

Write a markdown cell (English, 5–8 lines): source, number of images / classes / buildings, class balance, known issues (duplicates, broken, blur outliers), what the dataset should **not** be used for.

This becomes section 1 of RND-002.

In [ ]:
# TODO


**Dataset card**

> ...

---
## Oturum 4 — Özellikler ve sızıntısız split

Sıra her zaman şu: **split → (fit on train) → transform train + test**. Veri setinin tamamından bir şey öğrenen her adım (imputer, scaler, model) `Pipeline` içinde olur ve sadece `X_train` üzerinde `fit` edilir.

### Exercise 16 — Feature extraction

Write `features(path) -> np.ndarray`:

- HSV histogram, 8 bins per channel, normalized (24 values)
- `edge_density`, `blur`, `brightness`
- aspect ratio `width / height`

Build `X` (n_images × 28), `y = df_clean['label']`, `groups = df_clean['group']`. Print shapes.

In [ ]:
# TODO


### Exercise 17 — Group split vs random split

Make two splits (test_size=0.2, `random_state=SEED`):

1. `train_test_split(..., stratify=y)`
2. `GroupShuffleSplit` on `groups`

For each, count how many buildings appear in **both** train and test. Explain in one sentence why split 1 leaks.

Use split 2 from here on.

In [ ]:
from sklearn.model_selection import train_test_split, GroupShuffleSplit
# TODO


### Exercise 18 — Fit on train only — see the difference

Fit a `StandardScaler` on **all** of X, then on `X_train` only. Compare the learned `mean_` of the first 3 features.

One sentence: what information leaks in the first version?

In [ ]:
from sklearn.preprocessing import StandardScaler
# TODO


### Exercise 19 — The Pipeline

Build `Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])`.

Fit on train, print train and test accuracy.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
# TODO


### Exercise 20 — The floor: DummyClassifier

`DummyClassifier(strategy='most_frequent')` in the same setup. Its score is the **floor**: a model that does not beat it has learned nothing.

How far above the floor is your Pipeline?

In [ ]:
from sklearn.dummy import DummyClassifier
# TODO


---
## Oturum 5 — Değerlendirme ve model seçimi

Bu oturumda test setine bakma. Seçimi train üzerinde `GroupKFold` CV ile yap. Test seti Oturum 6'da bir kez kullanılacak.

### Exercise 21 — Cross-validated metrics

With `GroupKFold(n_splits=5)` on **train only**, get out-of-fold predictions (`cross_val_predict(..., groups=groups_train)`).

Print accuracy, macro precision / recall / F1 and `classification_report`. Which metric matters most for your classes, and why macro instead of weighted?

In [ ]:
from sklearn.model_selection import GroupKFold, cross_val_predict, cross_val_score
from sklearn.metrics import classification_report
# TODO


### Exercise 22 — Confusion matrix — interpret it

`ConfusionMatrixDisplay.from_predictions` on the out-of-fold predictions.

Name the most frequent confusion (A predicted as B) and give a visual hypothesis for it.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
# TODO


### Exercise 23 — Precision–recall curve

Pick one class as positive (one-vs-rest). Using out-of-fold `predict_proba`, plot the PR curve and print **average precision (AP)**.

One sentence: what does moving the threshold do to precision and recall? (In Modül 4, mAP is this AP averaged over classes.)

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay
# TODO


### Exercise 24 — Second model

Same Pipeline with `RandomForestClassifier(n_estimators=200, max_depth=6, random_state=SEED)` (no scaler needed, keep it for symmetry).

Compare both with `cross_val_score(..., groups=..., scoring='f1_macro')`: mean ± std.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
# TODO


### Exercise 25 — Choose — with the std in mind

Which model do you keep? If the difference in mean is smaller than the std across folds, say so. That is a valid result: "no reliable difference".

In [ ]:
# TODO


---
## Oturum 6 — R&D: deneyler, hata analizi, RND-002

Her deneyde tek değişken değişir. Tüm deneyler train + GroupKFold üzerinde yapılır.

### Exercise 26 — Error analysis grid

Show 12 misclassified images (out-of-fold) with `true → pred` titles.

Group the errors into 2–3 named failure types (e.g. 'night shots', 'tight crops'). Count each.

In [ ]:
# TODO


### Exercise 27 — Experiment 1 — feature set

A: HSV histogram only. B: HSV + edge/blur/brightness/aspect. Same model, same folds.

Fill the experiment table below.

In [ ]:
# TODO


### Exercise 28 — Experiment 2 — class_weight

`class_weight=None` vs `'balanced'` on your chosen model. Look at per-class recall, not just the mean.

In [ ]:
# TODO


### Exercise 29 — Final test — once

Fit the chosen Pipeline on the full train set, evaluate on the test set **once**. Report macro F1 and the confusion matrix.

Then, for comparison only, run the same Pipeline with the **random** split from Ex 17. How much higher is the leaky score? That gap is a headline finding for RND-002.

In [ ]:
# TODO


### Exercise 30 — Export the baseline + RND-002

Move `features()`, the Pipeline and the split into `src/m2_baseline.py` with a `main()` that prints the final metrics. It should run with `python src/m2_baseline.py`. Also save `X.npy`, `y.npy`, `groups.npy` to `outputs/m2/` — Modül 3 starts from them.

Copy `rnd/RND-000-template.md` to `rnd/RND-002-dataset-audit-baseline/report.md` and fill in: dataset card (Ex 15), baseline + floor, the two experiments, the leakage gap, failure types, next experiment (Modül 3: CNN vs this baseline).

In [ ]:
# TODO


---
## Oturum 7 — Sıfırdan: metrikler

> Metrics are a few lines of NumPy; writing them removes the mystery.

### Exercise 31 — Confusion matrix

From scratch for k classes; compare with sklearn.

In [ ]:
# TODO


**Gözlem:** 

### Exercise 32 — Precision/recall/F1

Per class, macro and weighted, from your confusion matrix.

In [ ]:
# TODO


**Gözlem:** 

### Exercise 33 — PR curve

From scores by sorting thresholds; AP as area; compare with `average_precision_score`.

In [ ]:
# TODO


**Gözlem:** 

### Exercise 34 — ROC vs PR

ROC-AUC from scratch; show a severely imbalanced case where ROC looks good and PR doesn't.

In [ ]:
# TODO


**Gözlem:** 

### Exercise 35 — From memory **(AI'sız)**

`confusion_matrix` + macro F1 from an empty cell, tested against sklearn.

In [ ]:
# TODO


**Gözlem:** 

### Experiment log

| Exp | Changed (one thing) | Baseline value | New value | Hypothesis | F1 macro (mean ± std) | Observation | Next |
|---|---|---|---|---|---|---|---|
| 1 | feature set | HSV only | HSV + quality |  |  |  |  |
| 2 | class_weight | None | balanced |  |  |  |  |
| 3 |  |  |  |  |  |  |  |

## Modül 2 kapanış kontrolü

Bir maddeyi ancak az ya da hiç AI yardımı olmadan yapabildiğinde işaretle.

- [ ] Bir görsel klasöründen manifest kurup bozuk dosya ve kopyaları yakalayabiliyorum
- [ ] `loc` / `iloc` / mask / `groupby` / `merge` arasındaki farkı açıklayabiliyorum
- [ ] Neyin sızıntı olduğunu, neyin olmadığını (görsel başına özellik vs. veri setinden fit) açıklayabiliyorum
- [ ] Group split'in neden gerekli olduğunu bir rakamla gösterebiliyorum (Ex 29)
- [ ] Bir Pipeline'ı train'e fit edip CV ile karşılaştırabiliyorum
- [ ] Confusion matrix ve PR eğrisini yorumlayabiliyorum
- [ ] `src/m2_baseline.py` sıfırdan çalışıyor
- [ ] RND-002 taslağı yazıldı, commit + push yapıldı, `PROGRESS.md` güncel

**Sonraki:** Modül 3, klasik ML derinleşme (`m03_classical_ml`). CNN'i bu baseline'a karşı koyma deneyi Modül 7'de (RND-005).